# 05 — Ensemble & Final Submission

Combines tuned models via rank blending and stacking. Applies pseudo-labeling. Generates final submission.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import roc_auc_score

from src.features import get_feature_cols, TARGET
from src.cv import run_cv, add_target_encoding
from src.models import make_lgb_fn, make_xgb_fn, make_cat_fn, train_full
from src.ensemble import blend_oof, blend_test, pseudo_label, stack_predict
from src.utils import rank_avg, optimise_weights, save_submission

cache_dir = Path('..') / 'cache'
train_feat = pd.read_pickle(cache_dir / 'train_feat.pkl')
test_feat  = pd.read_pickle(cache_dir / 'test_feat.pkl')
ext_feat   = pd.read_pickle(cache_dir / 'ext_feat.pkl')

with open(cache_dir / 'best_params.pkl', 'rb') as f:
    best_params = pickle.load(f)
with open(cache_dir / 'oof_preds_tuned.pkl', 'rb') as f:
    oof_tuned = pickle.load(f)

# Re-apply target encoding
train_feat, ext_feat, test_feat = add_target_encoding(train_feat, ext_feat, test_feat)
feature_cols = get_feature_cols(train_feat)
te_cols = [c for c in ['Driver_TE', 'Race_TE', 'Race_Year_TE'] if c in train_feat.columns]
feature_cols = feature_cols + te_cols

y_true = train_feat[TARGET].values
print(f'Loaded. Feature count: {len(feature_cols)}')

## 1. Generate test predictions with tuned full models

In [ ]:
X_tr_all = pd.concat([train_feat[feature_cols], ext_feat[feature_cols]]).values
y_tr_all = pd.concat([train_feat[TARGET], ext_feat[TARGET]]).values
w_tr_all = np.concatenate([np.ones(len(train_feat)), np.full(len(ext_feat), 0.8)])
X_test = test_feat[feature_cols].values

# Determine optimal n_estimators from CV (best fold avg × 1.05)
n_est_lgb = 1000  # conservative; adjust based on notebook 04 early stopping rounds
n_est_xgb = 1000
n_est_cat = 1000

print('Training full LGB (tuned)...')
lgb_full = train_full(X_tr_all, y_tr_all, w_tr_all, 'lgb', best_params['lgb'], n_est_lgb)
lgb_test = lgb_full.predict_proba(X_test)[:, 1]

print('Training full XGB (tuned)...')
xgb_full = train_full(X_tr_all, y_tr_all, w_tr_all, 'xgb', best_params['xgb'], n_est_xgb)
xgb_test = xgb_full.predict_proba(X_test)[:, 1]

print('Training full CatBoost (tuned)...')
cat_full = train_full(X_tr_all, y_tr_all, w_tr_all, 'cat', best_params['cat'], n_est_cat)
cat_test = cat_full.predict_proba(X_test)[:, 1]

print('Done.')

## 2. Rank-average ensemble

In [ ]:
lgb_oof = oof_tuned['lgb']
xgb_oof = oof_tuned['xgb']
cat_oof = oof_tuned['cat']

rank_oof = rank_avg([lgb_oof, xgb_oof, cat_oof])
rank_test = rank_avg([lgb_test, xgb_test, cat_test])
print(f'Rank ensemble OOF AUC: {roc_auc_score(y_true, rank_oof):.5f}')

## 3. Optimised weighted ensemble

In [ ]:
w_opt = optimise_weights([lgb_oof, xgb_oof, cat_oof], y_true)
print(f'Optimised weights: LGB={w_opt[0]:.3f}  XGB={w_opt[1]:.3f}  CAT={w_opt[2]:.3f}')
weighted_oof = sum(wi * p for wi, p in zip(w_opt, [lgb_oof, xgb_oof, cat_oof]))
weighted_test = sum(wi * p for wi, p in zip(w_opt, [lgb_test, xgb_test, cat_test]))
print(f'Weighted ensemble OOF AUC: {roc_auc_score(y_true, weighted_oof):.5f}')

## 4. Stacking (LR meta-learner)

In [ ]:
stacked_test = stack_predict(
    [lgb_oof, xgb_oof, cat_oof],
    [lgb_test, xgb_test, cat_test],
    y_true,
    train_feat,
)

## 5. Pseudo-labeling

In [ ]:
# Use rank ensemble test preds for pseudo-labeling
train_feat_pl, ext_feat_pl = pseudo_label(
    train_feat, ext_feat, test_feat, rank_test, feature_cols,
    high_thresh=0.90, low_thresh=0.05, pseudo_weight=0.5
)

# Retrain LGB with pseudo-labeled data
print('\nRunning CV with pseudo-labeled data (LGB)...')
best_lgb_fn = make_lgb_fn(best_params['lgb'])
lgb_oof_pl, _ = run_cv(train_feat_pl, ext_feat_pl, feature_cols, best_lgb_fn)
print(f'LGB OOF AUC (with pseudo-labels): {roc_auc_score(y_true, lgb_oof_pl):.5f}')

## 6. Final comparison and submission selection

In [ ]:
results = {
    'LGB tuned': roc_auc_score(y_true, lgb_oof),
    'XGB tuned': roc_auc_score(y_true, xgb_oof),
    'CAT tuned': roc_auc_score(y_true, cat_oof),
    'Rank ensemble': roc_auc_score(y_true, rank_oof),
    'Weighted ensemble': roc_auc_score(y_true, weighted_oof),
    'LGB + pseudo-labels': roc_auc_score(y_true, lgb_oof_pl),
}

print('=== Final OOF AUC Summary ===')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k:<28} {v:.5f}')

In [ ]:
# Use rank ensemble for final submission (most robust)
final_pred = rank_test

sub_path = save_submission(test_feat['id'], final_pred, tag='tuned_rank_ensemble_final')
print(f'\nFinal submission: {sub_path}')

# Also save stacked version
sub_path_stacked = save_submission(test_feat['id'], stacked_test, tag='stacked_final')
print(f'Stacked submission: {sub_path_stacked}')